In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('Data/Joined_Data.csv', low_memory=False)

In [3]:
data.columns

Index(['SiteKey', 'Timestamp', 'SolarGeneration', 'ApparentTemperature',
       'AirTemperature', 'DewPointTemperature', 'RelativeHumidity',
       'WindSpeed', 'WindDirection', 'kWp', 'Number of panels', 'Panel',
       'Inverter', 'Optimizers', 'Metric', 'lat', 'Lon'],
      dtype='object')

In [4]:
lat = np.deg2rad(data["lat"].iloc[0])
print(lat)

-0.6583082879845941


In [5]:
data.drop(columns=["Number of panels","Panel","Inverter","Optimizers","Metric","lat","Lon"], inplace=True)

In [6]:
data.columns

Index(['SiteKey', 'Timestamp', 'SolarGeneration', 'ApparentTemperature',
       'AirTemperature', 'DewPointTemperature', 'RelativeHumidity',
       'WindSpeed', 'WindDirection', 'kWp'],
      dtype='object')

In [7]:
data["Timestamp"] = pd.to_datetime(data["Timestamp"], errors="coerce")
data = data.sort_values(["SiteKey", "Timestamp"])

data["CapacityFactor"] = data["SolarGeneration"] / data["kWp"]

data["hour"] = data["Timestamp"].dt.hour
data["minute"] = data["Timestamp"].dt.minute
data["dayofyear"] = data["Timestamp"].dt.dayofyear
data["month"] = data["Timestamp"].dt.month

data["decl"] = 23.44 * np.pi/180 * np.sin(2 * np.pi * (data["dayofyear"] - 81) / 365)

cos_omega = -np.tan(lat) * np.tan(data["decl"])
cos_omega = np.clip(cos_omega, -1, 1)
omega = np.arccos(cos_omega)
day_length = 2 * omega * 24 / (2 * np.pi)

data["sunrise"] = 12 - day_length / 2
data["sunset"] = 12 + day_length / 2
data["time_float"] = data["hour"] + data["minute"] / 60

data["is_night"] = (
    (data["time_float"] < data["sunrise"]) |
    (data["time_float"] > data["sunset"])
)
data.loc[data["is_night"], "CapacityFactor"] = 0

def interpolate_day(cf, is_night):
    cf = cf.copy()
    cf[~is_night] = cf[~is_night].interpolate(limit=3)
    return cf

data["CapacityFactor"] = interpolate_day(
    data.groupby("SiteKey")["CapacityFactor"].transform(lambda x: x),
    data["is_night"]
)

data["CapacityFactor"] = (
    data.groupby(["SiteKey", "month", "hour"])["CapacityFactor"]
    .transform(lambda x: x.fillna(x.median()))
)

site_median = data.groupby("SiteKey")["CapacityFactor"].transform("median")
data["CapacityFactor"] = data["CapacityFactor"].fillna(site_median)
global_median = data["CapacityFactor"].median()
data["CapacityFactor"] = data["CapacityFactor"].fillna(global_median)

data["CapacityFactor"] = data["CapacityFactor"].clip(0, 1.2)
data["SolarGeneration"] = data["CapacityFactor"] * data["kWp"]

data = data.drop(columns=["minute","is_night","Timestamp","month","kWp","sunrise", "sunset"])

In [8]:
print(data.isnull().sum())

SiteKey                     0
SolarGeneration             0
ApparentTemperature    364774
AirTemperature         364774
DewPointTemperature    364774
RelativeHumidity       364774
WindSpeed              731786
WindDirection          731786
CapacityFactor              0
hour                        0
dayofyear                   0
decl                        0
time_float                  0
dtype: int64


In [9]:
weather_cols = ["ApparentTemperature", "AirTemperature","DewPointTemperature", "RelativeHumidity", 
"WindSpeed", "WindDirection"]

data[weather_cols] = data.groupby("SiteKey")[weather_cols] \
    .transform(lambda x: x.interpolate(limit=4))

data[weather_cols] = data.groupby(["SiteKey","hour"])[weather_cols] \
    .transform(lambda x: x.fillna(x.median()))

In [10]:
print(data.isnull().sum())

SiteKey                0
SolarGeneration        0
ApparentTemperature    0
AirTemperature         0
DewPointTemperature    0
RelativeHumidity       0
WindSpeed              0
WindDirection          0
CapacityFactor         0
hour                   0
dayofyear              0
decl                   0
time_float             0
dtype: int64


In [11]:
data["WindDir_sin"] = np.sin(np.deg2rad(data["WindDirection"]))
data["WindDir_cos"] = np.cos(np.deg2rad(data["WindDirection"]))

data["Hour_sin"] = np.sin(2*np.pi*data["hour"]/24)
data["Hour_cos"] = np.cos(2*np.pi*data["hour"]/24)

data["Day_sin"] = np.sin(2*np.pi*data["dayofyear"]/365)
data["Day_cos"] = np.cos(2*np.pi*data["dayofyear"]/365)

data = data.drop(columns=["WindDirection","dayofyear", "hour"])

In [12]:
data["hour_angle"] = (data["time_float"] - 12) * (np.pi / 12)
data["sin_altitude"] = (
    np.sin(lat) * np.sin(data["decl"]) +
    np.cos(lat) * np.cos(data["decl"]) * np.cos(data["hour_angle"])
)
data["SolarPotential"] = np.clip(data["sin_altitude"], 0, None)

data = data.drop(columns=["hour_angle", "sin_altitude", "time_float","decl"])

In [13]:
data["Temp_solar"] = data["AirTemperature"] * data["SolarPotential"]
data["Humidity_solar"] = data["RelativeHumidity"] * data["SolarPotential"]
data["Wind_solar"] = data["WindSpeed"] * data["SolarPotential"]

data["Temp_change"] = data.groupby("SiteKey")["AirTemperature"].diff()
data["Humidity_change"] = data.groupby("SiteKey")["RelativeHumidity"].diff()
data["Wind_change"] = data.groupby("SiteKey")["WindSpeed"].diff()

In [14]:
for i in range(1,4):
    data[f"Lag_{i}"] = data.groupby("SiteKey")["CapacityFactor"].shift(i)

data["Ramp_1"] = data["Lag_1"] - data["Lag_2"]
data["Ramp_2"] = data["Lag_2"] - data["Lag_3"]

data["RollingStd_4"] = (
    data.groupby("SiteKey")["CapacityFactor"]
    .shift(1)
    .rolling(4)
    .std()
)

data["CF_smooth_4"] = (
    data.groupby("SiteKey")["CapacityFactor"]
    .shift(1)
    .rolling(4)
    .mean()
)

data["CloudinessIndex"] = data["CF_smooth_4"] / (data["SolarPotential"] + 1e-6)
data["CloudinessIndex"] = np.clip(data["CloudinessIndex"],0,2)

data["CloudTrend"] = data.groupby("SiteKey")["CloudinessIndex"].diff()

In [15]:
data["Target_15min"] = data.groupby("SiteKey")["CapacityFactor"].shift(-1)
data["Target_1h"] = data.groupby("SiteKey")["CapacityFactor"].shift(-4)
data["Target_3h"] = data.groupby("SiteKey")["CapacityFactor"].shift(-12)

data["Residual_now"] = data["CapacityFactor"] - data["Lag_1"]
data["Residual_15min"] = data["Target_15min"] - data["Lag_1"]
data["Residual_1h"] = data["Target_1h"] - data["Lag_1"]
data["Residual_3h"] = data["Target_3h"] - data["Lag_1"]

In [16]:
print(data.isnull().sum())

SiteKey                  0
SolarGeneration          0
ApparentTemperature      0
AirTemperature           0
DewPointTemperature      0
RelativeHumidity         0
WindSpeed                0
CapacityFactor           0
WindDir_sin              0
WindDir_cos              0
Hour_sin                 0
Hour_cos                 0
Day_sin                  0
Day_cos                  0
SolarPotential           0
Temp_solar               0
Humidity_solar           0
Wind_solar               0
Temp_change             25
Humidity_change         25
Wind_change             25
Lag_1                   25
Lag_2                   50
Lag_3                   75
Ramp_1                  50
Ramp_2                  75
RollingStd_4           100
CF_smooth_4            100
CloudinessIndex        100
CloudTrend             125
Target_15min            25
Target_1h              100
Target_3h              300
Residual_now            25
Residual_15min          50
Residual_1h            125
Residual_3h            325
d

In [17]:
data.dropna(
    subset=[
        "Lag_1","Lag_2","Lag_3",
        "RollingStd_4","CF_smooth_4","CloudTrend",
        "Residual_15min","Residual_1h","Residual_3h"
    ],
    inplace=True
)

In [18]:
print(data.isnull().sum())

SiteKey                0
SolarGeneration        0
ApparentTemperature    0
AirTemperature         0
DewPointTemperature    0
RelativeHumidity       0
WindSpeed              0
CapacityFactor         0
WindDir_sin            0
WindDir_cos            0
Hour_sin               0
Hour_cos               0
Day_sin                0
Day_cos                0
SolarPotential         0
Temp_solar             0
Humidity_solar         0
Wind_solar             0
Temp_change            0
Humidity_change        0
Wind_change            0
Lag_1                  0
Lag_2                  0
Lag_3                  0
Ramp_1                 0
Ramp_2                 0
RollingStd_4           0
CF_smooth_4            0
CloudinessIndex        0
CloudTrend             0
Target_15min           0
Target_1h              0
Target_3h              0
Residual_now           0
Residual_15min         0
Residual_1h            0
Residual_3h            0
dtype: int64


In [19]:
data.to_csv('Data/Final_Data.csv',index=False)